In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, RobustScaler
import statsmodels.api as sm
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve
from scipy.stats import chi2_contingency

In [2]:
patients = pd.read_csv("../../data/processed/modeling/patients_readmission_flag.csv")
episodes = pd.read_csv("../../data/processed/modeling/inpatient_episodes.csv")
train_pat, test_pat = train_test_split(
    patients,
    test_size=0.2,
    random_state=42, 
    stratify=patients["readmission_flag"]
)

train_ids = set(train_pat["crypted_patient_id"])
test_ids  = set(test_pat["crypted_patient_id"])

train_df = episodes[episodes["crypted_patient_id"].isin(train_ids)].copy()
test_df  = episodes[episodes["crypted_patient_id"].isin(test_ids)].copy()

def preprocess_features(df):
    def map_age(group):
        nums = [int(s) for s in str(group).replace('+', '').split('-') if s.isdigit()]
        return sum(nums) / len(nums)
    df['age'] = df['age_group'].apply(map_age)
    # OHE for discharge_department
    df = pd.get_dummies(df, columns=['discharge_department'], prefix='dept', drop_first=True)
    return df
X_train = preprocess_features(train_df)
X_test = preprocess_features(test_df)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
y_train = X_train['is_readmitted']
X_train = X_train.drop(columns=['is_readmitted'])


# false positive 1 every 20 experiments (0.05 threshold for statistical significance - false positive threshold 5%)
clean_data = pd.concat([X_train, y_train], axis=1)
clean_data['12m_prior_meds'] = clean_data['P_ATC_12m_before'] + clean_data['S_ATC_12m_before'] + clean_data['D_ATC_12m_before']
clean_data['meds_during'] = clean_data['P_ATC_during_episode'] + clean_data['S_ATC_during_episode'] + clean_data['D_ATC_during_episode']
# Calculate Point-Biserial Correlation
correlation_age, p_value_age = stats.pointbiserialr(clean_data['is_readmitted'], clean_data['age'])
correlation_cci, p_value_cci = stats.pointbiserialr(clean_data['is_readmitted'], clean_data['age_adj_comorbidity_score'])
correlation__meds, p_value_meds = stats.pointbiserialr(clean_data['is_readmitted'], clean_data['num_medications_at_discharge'])
correlation_12m_meds, p_value_12m_meds = stats.pointbiserialr(clean_data['is_readmitted'], clean_data['12m_prior_meds'])
correlation_inpatient_visits, p_value_inpatient_visits = stats.pointbiserialr(clean_data['is_readmitted'], clean_data['count_prior_inpatient'])
correlation_los, p_value_los = stats.pointbiserialr(clean_data['is_readmitted'], clean_data['num_days'])
correlation_during_meds, p_value_during_meds = stats.pointbiserialr(clean_data['is_readmitted'], clean_data['meds_during'])
corr, p_value = stats.pointbiserialr(clean_data['age'], clean_data['num_medications_at_discharge'])
correlation_prior_ambulatory, p_value_prior_ambulatory = stats.pointbiserialr(clean_data['is_readmitted'], clean_data['count_prior_ambulatory'])
correlation_emergency_admission, p_value_emergency_admission = stats.pointbiserialr(clean_data['is_readmitted'], clean_data['count_prior_emergency'])
print(f"Correlation between Age and num_medications_at_discharge: {corr:.4f}")
print(f"p-value for correlation between Age and num_medications_at_discharge: {p_value:.4e}")
print(f"Point-Biserial Correlation (prior_ambulatory): {correlation_prior_ambulatory:.4f}")
print(f"p-value (prior_ambulatory): {p_value_prior_ambulatory:.4e}")
print(f"Point-Biserial Correlation (age): {correlation_age:.4f}")
print(f"p-value (age): {p_value_age:.4e}")
print(f"Point-Biserial Correlation (CCI): {correlation_cci:.4f}")
print(f"p-value (CCI): {p_value_cci:.4e}")
print(f"Point-Biserial Correlation (num_medications_at_discharge): {correlation__meds:.4f}")
print(f"p-value (num_medications_at_discharge): {p_value_meds:.4e}")
print(f"Point-Biserial Correlation (12m_prior_meds): {correlation_12m_meds:.4f}")
print(f"p-value (12m_prior_meds): {p_value_12m_meds:.4e}")
print(f"Point-Biserial Correlation (count_prior_inpatient): {correlation_inpatient_visits:.4f}")
print(f"p-value (count_prior_inpatient): {p_value_inpatient_visits:.4e}")
print(f"Point-Biserial Correlation (num_days): {correlation_los:.4f}")
print(f"p-value (num_days): {p_value_los:.4e}")
print(f"Point-Biserial Correlation (count_prior_emergency): {correlation_emergency_admission:.4f}")
print(f"p-value (count_prior_emergency): {p_value_emergency_admission:.4e}")
print(f"Point-Biserial Correlation (meds_during): {correlation_during_meds:.4f}")
print(f"p-value (meds_during): {p_value_during_meds:.4e}")

if p_value_age < 0.05:
    print("\nConclusion: The relationship between Age and Readmission is statistically significant.")
else:
    print("\nConclusion: The relationship between Age and Readmission is NOT statistically significant.")
if p_value_prior_ambulatory < 0.05:
    print("Conclusion: The relationship between count_prior_ambulatory and Readmission is statistically significant.")
else:
    print("Conclusion: The relationship between count_prior_ambulatory and Readmission is NOT statistically significant.")
if p_value_cci < 0.05:
    print("Conclusion: The relationship between CCI and Readmission is statistically significant.")
else:
    print("Conclusion: The relationship between CCI and Readmission is NOT statistically significant.")
if p_value_meds < 0.05:
    print("Conclusion: The relationship between num_medications_at_discharge and Readmission is statistically significant.")
else:
    print("Conclusion: The relationship between num_medications_at_discharge and Readmission is NOT statistically significant.")
if p_value_12m_meds < 0.05:
    print("Conclusion: The relationship between 12m_prior_meds and Readmission is statistically significant.")
else:
    print("Conclusion: The relationship between 12m_prior_meds and Readmission is NOT statistically significant.")
if p_value_inpatient_visits < 0.05:
    print("Conclusion: The relationship between count_prior_inpatient and Readmission is statistically significant.")
else:
    print("Conclusion: The relationship between count_prior_inpatient and Readmission is NOT statistically significant.")
if p_value_los < 0.05:
    print("Conclusion: The relationship between num_days and Readmission is statistically significant.")
else:
    print("Conclusion: The relationship between LOS and Readmission is NOT statistically significant.")
if p_value_emergency_admission < 0.05:
    print("Conclusion: The relationship between count_prior_emergency and Readmission is statistically significant.")
else:
    print("Conclusion: The relationship between count_prior_emergency and Readmission is NOT statistically significant.")
if p_value_during_meds < 0.05:
    print("Conclusion: The relationship between meds_during and Readmission is statistically significant.")
else:
    print("Conclusion: The relationship between meds_during and Readmission is NOT statistically significant.")
dept_ohe_columns = [
    'dept_Cardiologia', 'dept_Chirurgia Toracica', 'dept_Chirurgia Vascolare - Angiologia', 'dept_Geriatria', 'dept_Lungodegenti', 'dept_Medicina Fisica E Riabilitazione', 'dept_Medicina Generale', 'dept_Nefrologia', 'dept_Nefrologia (Abilitato Al Trapianto Rene)', 'dept_Neurologia', 'dept_Oncologia', 'dept_Pneumologia', 'dept_Terapia Intensiva', 'dept_Terapia Semi Intensiva', 'dept_Unità Coronarica'
]

results = []
for col in dept_ohe_columns:
    # Create the 2x2 contingency table: Feature (0 or 1) vs Target (0 or 1)
    contingency_table = pd.crosstab(clean_data[col], clean_data['is_readmitted'])

    chi2_stat, p_val, dof, expected = chi2_contingency(contingency_table)
 
    results.append({
        'Department_Feature': col,
        'Chi_Square_Stat': chi2_stat,
        'p_value': p_val,
        'Is_Significant': p_val < 0.05
    })

ohe_stats_df = pd.DataFrame(results).sort_values(by='p_value')

# Format for clean printing
pd.options.display.float_format = '{:.4e}'.format
print("\n--- CHI-SQUARE RESULTS FOR OHE DEPARTMENTS ---")
print(ohe_stats_df.to_string(index=False))

Correlation between Age and num_medications_at_discharge: 0.0675
p-value for correlation between Age and num_medications_at_discharge: 5.9982e-06
Point-Biserial Correlation (prior_ambulatory): 0.0546
p-value (prior_ambulatory): 2.5239e-04
Point-Biserial Correlation (age): -0.0137
p-value (age): 3.5989e-01
Point-Biserial Correlation (CCI): 0.0787
p-value (CCI): 1.2894e-07
Point-Biserial Correlation (num_medications_at_discharge): 0.0076
p-value (num_medications_at_discharge): 6.1042e-01
Point-Biserial Correlation (12m_prior_meds): 0.1075
p-value (12m_prior_meds): 5.1033e-13
Point-Biserial Correlation (count_prior_inpatient): 0.1235
p-value (count_prior_inpatient): 9.8798e-17
Point-Biserial Correlation (num_days): -0.0043
p-value (num_days): 7.7360e-01
Point-Biserial Correlation (count_prior_emergency): 0.1210
p-value (count_prior_emergency): 3.9694e-16
Point-Biserial Correlation (meds_during): 0.0156
p-value (meds_during): 2.9618e-01

Conclusion: The relationship between Age and Readmis